# Snowflake Data Quality & DMF – Scenario-Based Hands-on Lab

**Database:** `SF_SCENARIO_BASED_LEARNING`  **Schema (new):** `DATA_QUALITY`

This lab copies `PII_DATA.CUSTOMERS` and `PII_DATA.ORDERS` into a new schema and adds **known bad rows**.
Each check should then catch exactly those rows. The original tables are not changed.

Run the cells top to bottom. Every scenario has an **Expected** note, so you can check the result.

> Requirements: Enterprise edition or higher, and a role that owns the tables (e.g. ACCOUNTADMIN). DMF runs use serverless credits.

## 0. Setup – create the schema and lab tables

In [ ]:
%%sql -r setup_schema
CREATE SCHEMA IF NOT EXISTS SF_SCENARIO_BASED_LEARNING.DATA_QUALITY;

In [ ]:
%%sql -r setup_customers
CREATE OR REPLACE TABLE SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.CUSTOMERS AS
SELECT CUSTOMER_ID, FIRST_NAME, LAST_NAME, EMAIL, PHONE_NUMBER, CITY, STATE, ZIP_CODE,
       CREATED_AT, CURRENT_TIMESTAMP()::TIMESTAMP_LTZ AS LOAD_TS
FROM SF_SCENARIO_BASED_LEARNING.PII_DATA.CUSTOMERS;

In [ ]:
%%sql -r setup_orders
CREATE OR REPLACE TABLE SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS AS
SELECT ORDER_ID, CUSTOMER_ID, ORDER_DATE, ORDER_AMOUNT, ORDER_STATUS,
       CURRENT_TIMESTAMP()::TIMESTAMP_LTZ AS LOAD_TS
FROM SF_SCENARIO_BASED_LEARNING.PII_DATA.ORDERS;

### Add known bad rows
| Table | Bad row | Issue it simulates |
| --- | --- | --- |
| CUSTOMERS | 9001 | NULL email |
| CUSTOMERS | 9002 | duplicate email (copy of customer 1's email) |
| CUSTOMERS | 9003 | invalid email format |
| CUSTOMERS | 9004 | blank email `'   '` |
| ORDERS | 9101 | negative amount |
| ORDERS | 9102 | invalid status `UNKNOWN` |
| ORDERS | 9103 | orphan customer_id 999999 |
| ORDERS | 9104 | NULL customer_id |

In [ ]:
%%sql -r inject_customers
INSERT INTO SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.CUSTOMERS (CUSTOMER_ID, FIRST_NAME, EMAIL, LOAD_TS)
SELECT 9001,'NullEmail',NULL,CURRENT_TIMESTAMP() UNION ALL
SELECT 9002,'DupEmail',(SELECT EMAIL FROM SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.CUSTOMERS WHERE CUSTOMER_ID=(SELECT MIN(CUSTOMER_ID) FROM SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.CUSTOMERS)),CURRENT_TIMESTAMP() UNION ALL
SELECT 9003,'BadEmail','not-an-email',CURRENT_TIMESTAMP() UNION ALL
SELECT 9004,'BlankEmail','   ',CURRENT_TIMESTAMP();

In [ ]:
%%sql -r inject_orders
INSERT INTO SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS (ORDER_ID, CUSTOMER_ID, ORDER_DATE, ORDER_AMOUNT, ORDER_STATUS, LOAD_TS) VALUES
 (9101, 1,      CURRENT_DATE(), -50, 'NEW',     CURRENT_TIMESTAMP()),
 (9102, 2,      CURRENT_DATE(), 100, 'UNKNOWN', CURRENT_TIMESTAMP()),
 (9103, 999, CURRENT_DATE(), 100, 'NEW',     CURRENT_TIMESTAMP()),
 (9104, NULL,   CURRENT_DATE(), 100, 'NEW',     CURRENT_TIMESTAMP());

In [ ]:
%%sql -r row_counts
SELECT 'CUSTOMERS' t, COUNT(*) c FROM SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.CUSTOMERS
UNION ALL SELECT 'ORDERS', COUNT(*) FROM SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS;
-- Expected: CUSTOMERS = 204, ORDERS = 204

## Scenario 1 – Ad-hoc spot checks with system DMFs
*"The dashboard looks wrong, so check the table quickly."* System DMFs can be called directly, with no schedule.

In [ ]:
%%sql -r adhoc_checks
SELECT
  SNOWFLAKE.CORE.NULL_COUNT     (SELECT EMAIL FROM SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.CUSTOMERS) AS null_emails,
  SNOWFLAKE.CORE.BLANK_COUNT    (SELECT EMAIL FROM SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.CUSTOMERS) AS blank_emails,
  SNOWFLAKE.CORE.DUPLICATE_COUNT(SELECT EMAIL FROM SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.CUSTOMERS) AS duplicate_emails,
  SNOWFLAKE.CORE.UNIQUE_COUNT   (SELECT EMAIL FROM SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.CUSTOMERS) AS unique_emails,
  -- ROW_COUNT takes no arguments and can't be called ad hoc; it's only attached via ADD DATA METRIC FUNCTION ... ON ()
  (SELECT COUNT(*) FROM SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.CUSTOMERS)                          AS row_count;
-- Expected: null_emails >= 1, blank_emails >= 1, duplicate_emails >= 1, row_count = 204

In [ ]:
%%sql -r adhoc_stats
SELECT
  SNOWFLAKE.CORE.MIN   (SELECT ORDER_AMOUNT FROM SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS) AS min_amt,
  SNOWFLAKE.CORE.MAX   (SELECT ORDER_AMOUNT FROM SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS) AS max_amt,
  SNOWFLAKE.CORE.AVG   (SELECT ORDER_AMOUNT FROM SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS) AS avg_amt,
  SNOWFLAKE.CORE.STDDEV(SELECT ORDER_AMOUNT FROM SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS) AS stddev_amt,
  SNOWFLAKE.CORE.NULL_COUNT(SELECT CUSTOMER_ID FROM SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS) AS null_cust_ids;
-- Expected: min_amt = -50 (bad row 9101), null_cust_ids = 1 (bad row 9104)

## Scenario 2 – Data freshness
*"The finance SLA says ORDERS must be less than 1 hour old."*
`FRESHNESS` returns the **seconds** between the newest timestamp and the DMF's scheduled time.

In [ ]:
%%sql -r adhoc_freshness
SELECT
  SNOWFLAKE.CORE.FRESHNESS(SELECT LOAD_TS FROM SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS) AS seconds_since_last_load;
-- Expected: a small number (the data was just loaded)

In [ ]:
%%sql -r metadata_freshness
-- Alternative without DMFs: table metadata (note: DDL also changes LAST_ALTERED)
SELECT TABLE_NAME, LAST_ALTERED, DATEDIFF('minute', LAST_ALTERED, CURRENT_TIMESTAMP()) AS minutes_old
FROM SF_SCENARIO_BASED_LEARNING.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'DATA_QUALITY';

## Scenario 3 – Custom DMFs for business rules
*"Emails must be valid, and refunds must never be booked as negative orders."*

In [ ]:
%%sql -r custom_dmf_email
CREATE OR REPLACE DATA METRIC FUNCTION SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.INVALID_EMAIL_COUNT(
  t TABLE(email STRING))
RETURNS NUMBER AS
$$
  SELECT COUNT_IF(email IS NOT NULL AND TRIM(email) <> ''
                  AND NOT REGEXP_LIKE(email, '^[^@ ]+@[^@ ]+\\.[^@ ]+$'))
  FROM t
$$;

In [ ]:
%%sql -r custom_dmf_negative
CREATE OR REPLACE DATA METRIC FUNCTION SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.NEGATIVE_AMOUNT_COUNT(
  t TABLE(amount NUMBER))
RETURNS NUMBER AS
$$ SELECT COUNT_IF(amount < 0) FROM t $$;

In [ ]:
%%sql -r custom_dmf_orphan
-- Two-table custom DMF: orders whose customer does not exist
CREATE OR REPLACE DATA METRIC FUNCTION SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORPHAN_CUSTOMER_COUNT(
  child TABLE(customer_id NUMBER), parent TABLE(customer_id NUMBER))
RETURNS NUMBER AS
$$
  SELECT COUNT(*) FROM child c
  WHERE c.customer_id IS NOT NULL
    AND NOT EXISTS (SELECT 1 FROM parent p WHERE p.customer_id = c.customer_id)
$$;

In [ ]:
%%sql -r custom_dmf_test
SELECT
  SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.INVALID_EMAIL_COUNT(SELECT EMAIL FROM SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.CUSTOMERS) AS invalid_emails,
  SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.NEGATIVE_AMOUNT_COUNT(SELECT ORDER_AMOUNT FROM SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS) AS negative_amounts;
-- Expected: invalid_emails >= 1 (row 9003), negative_amounts = 1 (row 9101)

## Scenario 4 – Schedule + attach DMFs (continuous monitoring)
The schedule is a **table-level** parameter shared by all DMFs on that table.
Options: `'5 MINUTE'` (also 15/30/60/720/1440), `'USING CRON 0 6 * * * UTC'`, `'TRIGGER_ON_CHANGES'`.

In [ ]:
%%sql -r set_schedule
ALTER TABLE SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.CUSTOMERS SET DATA_METRIC_SCHEDULE = '5 MINUTE';

In [ ]:
%%sql -r set_schedule_orders
ALTER TABLE SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS SET DATA_METRIC_SCHEDULE = '5 MINUTE';

In [ ]:
%%sql -r attach_customers
-- One ADD DATA METRIC FUNCTION clause, DMFs comma-separated
ALTER TABLE SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.CUSTOMERS
  ADD DATA METRIC FUNCTION
    SNOWFLAKE.CORE.NULL_COUNT      ON (EMAIL),
    SNOWFLAKE.CORE.BLANK_COUNT     ON (EMAIL),
    SNOWFLAKE.CORE.DUPLICATE_COUNT ON (EMAIL),
    SNOWFLAKE.CORE.ROW_COUNT       ON (),
    SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.INVALID_EMAIL_COUNT ON (EMAIL);

In [ ]:
%%sql -r attach_orders
-- One ADD DATA METRIC FUNCTION clause, DMFs comma-separated
ALTER TABLE SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS
  ADD DATA METRIC FUNCTION
    SNOWFLAKE.CORE.FRESHNESS  ON (LOAD_TS),
    SNOWFLAKE.CORE.ROW_COUNT  ON (),
    SNOWFLAKE.CORE.NULL_COUNT ON (CUSTOMER_ID),
    SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.NEGATIVE_AMOUNT_COUNT ON (ORDER_AMOUNT),
    SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORPHAN_CUSTOMER_COUNT
      ON (CUSTOMER_ID, TABLE(SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.CUSTOMERS(CUSTOMER_ID)));

In [ ]:
%%sql -r attached_dmfs
SELECT METRIC_DATABASE_NAME||'.'||METRIC_SCHEMA_NAME||'.'||METRIC_NAME AS dmf,
       REF_ARGUMENTS, SCHEDULE, SCHEDULE_STATUS
FROM TABLE(SF_SCENARIO_BASED_LEARNING.INFORMATION_SCHEMA.DATA_METRIC_FUNCTION_REFERENCES(
  REF_ENTITY_NAME => 'SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS', REF_ENTITY_DOMAIN => 'TABLE'));
-- Expected: 5 rows with SCHEDULE_STATUS = STARTED

## Scenario 5 – Validity: accepted values
*"ORDER_STATUS must be NEW, SHIPPED, DELIVERED or RETURNED."* `ACCEPTED_VALUES` counts rows **not** in the list.
If your account doesn't have this DMF yet, skip this cell and use the plain SQL check after it.

In [ ]:
%%sql -r attach_accepted_values
ALTER TABLE SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS
  ADD DATA METRIC FUNCTION SNOWFLAKE.CORE.ACCEPTED_VALUES
  ON (ORDER_STATUS, ORDER_STATUS -> ORDER_STATUS IN ('NEW','SHIPPED','DELIVERED','RETURNED'));

In [ ]:
%%sql -r accepted_values_manual
SELECT COUNT(*) AS invalid_status_rows
FROM SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS
WHERE ORDER_STATUS NOT IN ('NEW','SHIPPED','DELIVERED','RETURNED') OR ORDER_STATUS IS NULL;
-- Expected: 1 (row 9102)

## Scenario 6 – Expectations (pass / fail)
An expectation turns a metric into PASS or FAIL. Here: no null emails, no negative amounts, and freshness under 1 hour.

In [ ]:
%%sql -r exp_null_email
ALTER TABLE SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.CUSTOMERS
  MODIFY DATA METRIC FUNCTION SNOWFLAKE.CORE.NULL_COUNT ON (EMAIL)
  ADD EXPECTATION no_null_email (VALUE = 0);

In [ ]:
%%sql -r exp_negative
ALTER TABLE SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS
  MODIFY DATA METRIC FUNCTION SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.NEGATIVE_AMOUNT_COUNT ON (ORDER_AMOUNT)
  ADD EXPECTATION no_negative_amount (VALUE = 0);

In [ ]:
%%sql -r exp_freshness
ALTER TABLE SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS
  MODIFY DATA METRIC FUNCTION SNOWFLAKE.CORE.FRESHNESS ON (LOAD_TS)
  ADD EXPECTATION fresh_within_1h (VALUE < 3600);

In [ ]:
%%sql -r evaluate_expectations
SELECT * FROM TABLE(SYSTEM$EVALUATE_DATA_QUALITY_EXPECTATIONS(
  REF_ENTITY_NAME => 'SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS'));
-- Expected: no_negative_amount = violated, fresh_within_1h = passing

## Scenario 7 – Read the monitoring results
⏳ Scheduled DMFs run about every 5 minutes. **Wait 5–10 minutes** after Scenario 4, then run these cells.
If they return no rows, wait a bit longer and rerun.

In [ ]:
%%sql -r results_orders
SELECT MEASUREMENT_TIME, METRIC_NAME, ARGUMENT_NAMES, VALUE
FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
  REF_ENTITY_NAME   => 'SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS',
  REF_ENTITY_DOMAIN => 'TABLE'))
ORDER BY MEASUREMENT_TIME DESC;

In [ ]:
%%sql -r results_all
SELECT TABLE_NAME, METRIC_NAME, ARGUMENT_NAMES, VALUE, MEASUREMENT_TIME
FROM SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS
WHERE TABLE_DATABASE = 'SF_SCENARIO_BASED_LEARNING' AND TABLE_SCHEMA = 'DATA_QUALITY'
QUALIFY ROW_NUMBER() OVER (PARTITION BY TABLE_NAME, METRIC_NAME, ARGUMENT_NAMES::STRING ORDER BY MEASUREMENT_TIME DESC) = 1
ORDER BY TABLE_NAME, METRIC_NAME;
-- Latest value per metric. Expected: NULL_COUNT(EMAIL) >= 1, NEGATIVE_AMOUNT_COUNT = 1, ORPHAN_CUSTOMER_COUNT = 1

## Scenario 8 – Find the failing rows (`SYSTEM$DATA_METRIC_SCAN`)
*"NULL_COUNT says 1 – which row is it?"* This works for row-level system DMFs such as NULL, BLANK and DUPLICATE.

In [ ]:
%%sql -r scan_null_email
SELECT * FROM TABLE(SYSTEM$DATA_METRIC_SCAN(
  REF_ENTITY_NAME => 'SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.CUSTOMERS',
  METRIC_NAME     => 'SNOWFLAKE.CORE.NULL_COUNT',
  ARGUMENT_NAME   => 'EMAIL'));
-- Expected: customer 9001

In [ ]:
%%sql -r scan_duplicate_email
SELECT * FROM TABLE(SYSTEM$DATA_METRIC_SCAN(
  REF_ENTITY_NAME => 'SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.CUSTOMERS',
  METRIC_NAME     => 'SNOWFLAKE.CORE.DUPLICATE_COUNT',
  ARGUMENT_NAME   => 'EMAIL'));
-- Expected: the original customer and 9002

## Scenario 9 – Freshness going stale (simulate a broken pipeline)
Move LOAD_TS back by 3 hours to simulate a load that stopped. The next FRESHNESS run should be over 10800 seconds,
and `fresh_within_1h` should fail.

In [ ]:
%%sql -r simulate_stale
UPDATE SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS SET LOAD_TS = DATEADD('hour', -3, LOAD_TS);

In [ ]:
%%sql -r check_stale
SELECT SNOWFLAKE.CORE.FRESHNESS(SELECT LOAD_TS FROM SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS) AS seconds_old;
-- Expected: about 10800 or more

## Scenario 10 – Circuit breaker (stop bad data reaching the next layer)
This procedure fails the pipeline step when bad data is found. In a task graph, child tasks won't run after the failure.

In [ ]:
%%sql -r circuit_breaker_proc
CREATE OR REPLACE PROCEDURE SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.DQ_GATE()
RETURNS STRING
LANGUAGE SQL
AS
$$
DECLARE
  neg NUMBER;
  nul NUMBER;
  dq_failed EXCEPTION (-20001, 'DQ gate failed: negative amounts or NULL customer_id found in ORDERS');
BEGIN
  -- SELECT ... INTO isn't allowed around a DMF call, so assign the result directly
  neg := (SELECT SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.NEGATIVE_AMOUNT_COUNT(
            SELECT ORDER_AMOUNT FROM SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS));
  nul := (SELECT SNOWFLAKE.CORE.NULL_COUNT(
            SELECT CUSTOMER_ID FROM SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS));
  IF (neg > 0 OR nul > 0) THEN
    RAISE dq_failed;
  END IF;
  RETURN 'DQ gate passed';
END;
$$;

In [ ]:
%%sql -r run_gate_fail
CALL SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.DQ_GATE();
-- Expected: ERROR "DQ gate failed..." (that is the point – the bad rows are still there)

In [ ]:
%%sql -r fix_data
DELETE FROM SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS WHERE ORDER_AMOUNT < 0 OR CUSTOMER_ID IS NULL;

In [ ]:
%%sql -r run_gate_pass
CALL SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.DQ_GATE();
-- Expected: 'DQ gate passed'

Wire it into a task graph like this (shown for reference, not run here):
```sql
CREATE TASK DQ_GATE_TASK WAREHOUSE=COMPUTE_WH SCHEDULE='USING CRON 0 6 * * * UTC'
  AS CALL SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.DQ_GATE();
CREATE TASK LOAD_GOLD WAREHOUSE=COMPUTE_WH AFTER DQ_GATE_TASK AS INSERT INTO ...;
```

## Scenario 11 – Alert on stale data (email)
Shown for reference. It needs an email notification integration, and the recipient must be a **verified** Snowflake user email.
```sql
CREATE NOTIFICATION INTEGRATION DQ_EMAIL_INT TYPE=EMAIL ENABLED=TRUE ALLOWED_RECIPIENTS=('you@company.com');

CREATE OR REPLACE ALERT SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS_STALE_ALERT
  WAREHOUSE = COMPUTE_WH
  SCHEDULE  = '60 MINUTE'
  IF (EXISTS (
      SELECT 1 FROM SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS
      WHERE TABLE_DATABASE='SF_SCENARIO_BASED_LEARNING' AND TABLE_SCHEMA='DATA_QUALITY'
        AND TABLE_NAME='ORDERS' AND METRIC_NAME='FRESHNESS'
        AND VALUE > 3600 AND MEASUREMENT_TIME > DATEADD('hour',-1,CURRENT_TIMESTAMP())))
  THEN CALL SYSTEM$SEND_EMAIL('DQ_EMAIL_INT','you@company.com','ORDERS is stale','Freshness > 1 hour');
ALTER ALERT SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS_STALE_ALERT RESUME;
```

## Scenario 12 – Manage DMFs: suspend, resume, drop

In [ ]:
%%sql -r suspend_dmf
ALTER TABLE SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.CUSTOMERS
  MODIFY DATA METRIC FUNCTION SNOWFLAKE.CORE.ROW_COUNT ON () SUSPEND;

In [ ]:
%%sql -r resume_dmf
ALTER TABLE SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.CUSTOMERS
  MODIFY DATA METRIC FUNCTION SNOWFLAKE.CORE.ROW_COUNT ON () RESUME;

In [ ]:
%%sql -r list_system_dmfs
SHOW DATA METRIC FUNCTIONS IN SCHEMA SNOWFLAKE.CORE;
-- Every system DMF available in your account

## Scenario 13 – Cost of DMFs (serverless credits)

In [ ]:
%%sql -r dmf_cost
SELECT START_TIME, END_TIME, CREDITS_USED, TABLE_NAME
FROM SNOWFLAKE.ACCOUNT_USAGE.DATA_QUALITY_MONITORING_USAGE_HISTORY
WHERE START_TIME > DATEADD('day', -7, CURRENT_TIMESTAMP())
ORDER BY START_TIME DESC;
-- ACCOUNT_USAGE can lag by up to a few hours

## Cleanup
Scheduled DMFs keep using credits every 5 minutes. When you've finished the lab, **run this cell** to stop them.

In [ ]:
%%sql -r cleanup_unschedule
ALTER TABLE SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.CUSTOMERS UNSET DATA_METRIC_SCHEDULE;

In [ ]:
%%sql -r cleanup_unschedule_orders
ALTER TABLE SF_SCENARIO_BASED_LEARNING.DATA_QUALITY.ORDERS UNSET DATA_METRIC_SCHEDULE;

To remove the lab completely (this drops only the lab schema, not `PII_DATA`):
```sql
DROP SCHEMA IF EXISTS SF_SCENARIO_BASED_LEARNING.DATA_QUALITY;
```